In [24]:
import ee

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library with your new dedicated project ID.
ee.Initialize(project='flood-extent-mapping-punjab')

print("Earth Engine Initialized Successfully! You are connected to the cloud.")

Earth Engine Initialized Successfully! You are connected to the cloud.


In [25]:
import ee

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library with your new dedicated project ID.
ee.Initialize(project='flood-extent-mapping-punjab')

print("Earth Engine Initialized Successfully! You are connected to the cloud.")


Earth Engine Initialized Successfully! You are connected to the cloud.


In [26]:
import ee

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library with your new dedicated project ID.
ee.Initialize(project='flood-extent-mapping-punjab')

print("Earth Engine Initialized Successfully! You are connected to the cloud.")

Earth Engine Initialized Successfully! You are connected to the cloud.


In [27]:
import ee

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library with your new dedicated project ID.
ee.Initialize(project='flood-extent-mapping-punjab')

print("Earth Engine Initialized Successfully! You are connected to the cloud.")


Earth Engine Initialized Successfully! You are connected to the cloud.


In [28]:
# 1. Define the Region of Interest: Punjab, Pakistan
# We are creating a rough bounding box geometry for the province.
punjab_roi = ee.Geometry.Polygon(
  [[[69.3, 27.7],
    [75.3, 27.7],
    [75.3, 34.0],
    [69.3, 34.0]]]
)

# 2. Define Timeframes for the 2022 Floods
# Pre-flood (Baseline normal conditions / Dry season)
pre_start = '2022-04-01'
pre_end = '2022-06-30'

# Post-flood (Peak inundation / Disaster phase)
post_start = '2022-08-15'
post_end = '2022-09-30'

print("Punjab ROI and 2022 Flood Timeframes successfully defined.")

Punjab ROI and 2022 Flood Timeframes successfully defined.


In [29]:
# 1. Access the Sentinel-1 Ground Range Detected (GRD) Image Collection
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(punjab_roi) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarization', 'VV')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarization', 'VH')) \
    .filter(ee.Filter.eq('instrumentMode', 'IW'))

# 2. Separate into Pre-Flood (Dry Season) and Post-Flood (Peak Disaster) Collections
pre_flood_coll = s1_collection.filterDate(pre_start, pre_end)
post_flood_coll = s1_collection.filterDate(post_start, post_end)

# 3. Create a Composite by taking the Median value of pixels (clears out extreme noise)
# We select the VV polarization band as it is highly sensitive to open water surfaces.
pre_flood_img = pre_flood_coll.select('VV').median().clip(punjab_roi)
post_flood_img = post_flood_coll.select('VV').median().clip(punjab_roi)

# 4. Print verification metrics to ensure the cloud actually retrieved files
print("Sentinel-1 SAR Data Fetched Successfully!")
print(f"Number of images in Pre-Flood collection: {pre_flood_coll.size().getInfo()}")
print(f"Number of images in Post-Flood collection: {post_flood_coll.size().getInfo()}")

Sentinel-1 SAR Data Fetched Successfully!
Number of images in Pre-Flood collection: 0
Number of images in Post-Flood collection: 0


In [30]:
# 1. Access the Sentinel-1 collection with looser, more stable filters
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(punjab_roi) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarization', 'VV'))

# 2. Re-apply the Pre-Flood and Post-Flood Date Ranges
pre_flood_coll = s1_collection.filterDate(pre_start, pre_end)
post_flood_coll = s1_collection.filterDate(post_start, post_end)

# 3. Create a Median Composite and clip it to our region
# (This step will now have actual data to process!)
pre_flood_img = pre_flood_coll.select('VV').median().clip(punjab_roi)
post_flood_img = post_flood_coll.select('VV').median().clip(punjab_roi)

# 4. Print the new verification metrics
print("Updated Sentinel-1 SAR Query Executed:")
print(f"Number of images in Pre-Flood collection: {pre_flood_coll.size().getInfo()}")
print(f"Number of images in Post-Flood collection: {post_flood_coll.size().getInfo()}")

Updated Sentinel-1 SAR Query Executed:
Number of images in Pre-Flood collection: 0
Number of images in Post-Flood collection: 0


In [31]:
import ee

# 1. Define a solid point location in Punjab, Pakistan (Latitude/Longitude)
punjab_point = ee.Geometry.Point([74.3587, 31.5204])

# 2. Access the Sentinel-1 collection using the point location
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(punjab_point) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarization', 'VV'))

# 3. Apply your 2022 Pre-Flood and Post-Flood Date Ranges
pre_flood_coll = s1_collection.filterDate(pre_start, pre_end)
post_flood_coll = s1_collection.filterDate(post_start, post_end)

# 4. Create the Median Composites using the point intersection
pre_flood_img = pre_flood_coll.select('VV').median().clip(punjab_point.buffer(50000)) # 50km radius
post_flood_img = post_flood_coll.select('VV').median().clip(punjab_point.buffer(50000))

# 5. Print the updated verification metrics
print("Point-Based Sentinel-1 SAR Query Executed:")
print(f"Number of images in Pre-Flood collection: {pre_flood_coll.size().getInfo()}")
print(f"Number of images in Post-Flood collection: {post_flood_coll.size().getInfo()}")

Point-Based Sentinel-1 SAR Query Executed:
Number of images in Pre-Flood collection: 0
Number of images in Post-Flood collection: 0


In [32]:
import ee

# 1. Point location in Punjab, Pakistan
punjab_point = ee.Geometry.Point([74.3587, 31.5204])

# 2. Access Sentinel-1 collection with ONLY spatial constraints
# Removing instrumentMode and polarization filters completely fixes the 0 image output.
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(punjab_point)

# 3. Apply your 2022 Pre-Flood and Post-Flood Date Ranges
pre_flood_coll = s1_collection.filterDate(pre_start, pre_end)
post_flood_coll = s1_collection.filterDate(post_start, post_end)

# 4. Create Median Composites dynamically picking the first available band
pre_flood_img = pre_flood_coll.median().clip(punjab_point.buffer(50000)) # 50km radius
post_flood_img = post_flood_coll.median().clip(punjab_point.buffer(50000))

# 5. Print the verification metrics
print("Broadened Sentinel-1 SAR Query Executed:")
print(f"Number of images found in Pre-Flood collection: {pre_flood_coll.size().getInfo()}")
print(f"Number of images found in Post-Flood collection: {post_flood_coll.size().getInfo()}")

Broadened Sentinel-1 SAR Query Executed:
Number of images found in Pre-Flood collection: 21
Number of images found in Post-Flood collection: 12


In [33]:
# 1. Define Training Points directly from the collected imagery spectrum
# We sample extreme values to train the model on what is definitely water vs. land.
# Water has a very low backscatter (low reflection), Land has high backscatter.

# Create sample regions (approximating water and land characteristics)
water_training = pre_flood_img.updateMask(pre_flood_img.lt(-16)) \
    .sample(region=punjab_point.buffer(50000), scale=30, numPixels=100, geometries=True) \
    .map(lambda f: f.set('landcover', 0)) # Class 0 = Water

land_training = pre_flood_img.updateMask(pre_flood_img.gt(-12)) \
    .sample(region=punjab_point.buffer(50000), scale=30, numPixels=100, geometries=True) \
    .map(lambda f: f.set('landcover', 1)) # Class 1 = Land

# Combine water and land training points into a single feature collection
training_features = water_training.merge(land_training)

# 2. Train the Random Forest Classifier Model
# We train it using 100 decision trees on our 'VV' radar band
classifier = ee.Classifier.smileRandomForest(100).train(
    features=training_features,
    classProperty='landcover',
    inputProperties=['VV']
)

# 3. Apply the trained Machine Learning Model to classify both periods
pre_classified = pre_flood_img.select('VV').classify(classifier)
post_classified = post_flood_img.select('VV').classify(classifier)

# 4. Isolate the Flood Footprint
# If a pixel was Land (1) before, but became Water (0) after, it is classified as FLOODED.
flood_mask = pre_classified.eq(1).And(post_classified.eq(0))

print("Random Forest Classifier trained and applied successfully!")

Random Forest Classifier trained and applied successfully!


In [34]:
# 1. Calculate the area of each individual flooded pixel (in square meters)
# ee.Image.pixelArea() dynamically handles projection distortions.
flood_area_image = flood_mask.multiply(ee.Image.pixelArea())

# 2. Sum up all the flooded pixels within our 50km analysis zone
stats = flood_area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=punjab_point.buffer(50000),
    scale=30, # Sentinel-1 resolution baseline
    maxPixels=1e9
)

# 3. Extract the area value and convert it from square meters to square kilometers
flooded_m2 = stats.get('VV') # Fetches the sum calculation from the radar band channel
flooded_km2 = ee.Number(flooded_m2).divide(1e6)

# 4. Print the final statistical data
print("Spatial Metrics Calculation Complete:")
print(f"Total Flooded Area within the 50km radius: {flooded_km2.getInfo():.2f} sq km")


Spatial Metrics Calculation Complete:


EEException: Classifier training failed: 'Invalid minimum size of leaf nodes: 1'.

In [ ]:
import ee

# 1. Access Sentinel-1 collection with ONLY spatial constraints (Confirmed 33 images)
punjab_point = ee.Geometry.Point([74.3587, 31.5204])
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(punjab_point)

pre_flood_coll = s1_collection.filterDate(pre_start, pre_end)
post_flood_coll = s1_collection.filterDate(post_start, post_end)

# Create Median Composites
pre_flood_img = pre_flood_coll.median().clip(punjab_point.buffer(50000))
post_flood_img = post_flood_coll.median().clip(punjab_point.buffer(50000))

# 2. Adjust sample thresholds to ensure a healthy array structure for the ML model
water_training = pre_flood_img.updateMask(pre_flood_img.lt(-15)) \
    .sample(region=punjab_point.buffer(50000), scale=30, numPixels=150, geometries=True) \
    .map(lambda f: f.set('landcover', 0))

land_training = pre_flood_img.updateMask(pre_flood_img.gt(-13)) \
    .sample(region=punjab_point.buffer(50000), scale=30, numPixels=150, geometries=True) \
    .map(lambda f: f.set('landcover', 1))

training_features = water_training.merge(land_training)

# 3. Explicitly configure the decision tree parameters to fix the Leaf Node Error
# Parameters passed: numberOfTrees=100, minLeafPopulation=2 (forces a valid split size)
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    minLeafPopulation=2
).train(
    features=training_features,
    classProperty='landcover',
    inputProperties=['VV']
)

# 4. Apply the machine learning model
pre_classified = pre_flood_img.select('VV').classify(classifier)
post_classified = post_flood_img.select('VV').classify(classifier)
flood_mask = pre_classified.eq(1).And(post_classified.eq(0))

# 5. Calculate Total Flooded Area in Square Kilometers
flood_area_image = flood_mask.multiply(ee.Image.pixelArea())
stats = flood_area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=punjab_point.buffer(50000),
    scale=30,
    maxPixels=1e9
)

flooded_m2 = stats.get('VV')
flooded_km2 = ee.Number(flooded_m2).divide(1e6)

print("Spatial Metrics Calculation Complete:")
print(f"Total Flooded Area within the 50km radius: {flooded_km2.getInfo():.2f} sq km")

In [ ]:
import ee

# 1. Point location in Punjab, Pakistan
punjab_point = ee.Geometry.Point([74.3587, 31.5204])
analysis_zone = punjab_point.buffer(50000) # 50km area

# 2. Access Sentinel-1 collection with ONLY spatial constraints
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(punjab_point)

pre_flood_coll = s1_collection.filterDate(pre_start, pre_end)
post_flood_coll = s1_collection.filterDate(post_start, post_end)

# Create Median Composites clipped to our zone
pre_flood_img = pre_flood_coll.select('VV').median().clip(analysis_zone)
post_flood_img = post_flood_coll.select('VV').median().clip(analysis_zone)

# 3. Calculate Dynamic Percentiles to find water vs. land automatically
# This ensures we grab actual samples for BOTH classes to prevent the "Only one class" error.
percentiles = pre_flood_img.reduceRegion(
    reducer=ee.Reducer.percentile([5, 95]),
    geometry=analysis_zone,
    scale=100,
    maxPixels=1e7
)

water_threshold = ee.Number(percentiles.get('VV_p5'))   # Darkest 5% of pixels
land_threshold = ee.Number(percentiles.get('VV_p95'))  # Brightest 5% of pixels

# 4. Generate Training Points using the dynamic thresholds
water_training = pre_flood_img.updateMask(pre_flood_img.lte(water_threshold)) \
    .sample(region=analysis_zone, scale=30, numPixels=100, geometries=True) \
    .map(lambda f: f.set('landcover', 0))

land_training = pre_flood_img.updateMask(pre_flood_img.gte(land_threshold)) \
    .sample(region=analysis_zone, scale=30, numPixels=100, geometries=True) \
    .map(lambda f: f.set('landcover', 1))

# Combine datasets safely
training_features = water_training.merge(land_training)

# 5. Train Random Forest Classifier
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    minLeafPopulation=2
).train(
    features=training_features,
    classProperty='landcover',
    inputProperties=['VV']
)

# 6. Run Machine Learning Classification and Extract Flood Mask
pre_classified = pre_flood_img.classify(classifier)
post_classified = post_flood_img.classify(classifier)

# Flood Mask logic: Was Land (1) before, turned into Water (0) after
flood_mask = pre_classified.eq(1).And(post_classified.eq(0))

# 7. Calculate Total Flooded Area in Square Kilometers
flood_area_image = flood_mask.multiply(ee.Image.pixelArea())
stats = flood_area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=analysis_zone,
    scale=30,
    maxPixels=1e9
)

flooded_m2 = stats.get('sum') # Corrected: 'VV' to 'sum'
flooded_km2 = ee.Number(flooded_m2).divide(1e6)

print("Spatial Metrics Calculation Complete:")
print(f"Total Flooded Area within the 50km radius: {flooded_km2.getInfo():.2f} sq km")

In [ ]:
import ee

# 1. Point location in Punjab, Pakistan
punjab_point = ee.Geometry.Point([74.3587, 31.5204])
analysis_zone = punjab_point.buffer(50000) # 50km area

# 2. Access Sentinel-1 collection with spatial constraints
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(punjab_point)

pre_flood_coll = s1_collection.filterDate(pre_start, pre_end)
post_flood_coll = s1_collection.filterDate(post_start, post_end)

# Create Median Composites clipped to our zone
pre_flood_img = pre_flood_coll.select('VV').median().clip(analysis_zone)
post_flood_img = post_flood_coll.select('VV').median().clip(analysis_zone)

# 3. Calculate Dynamic Percentiles to balance water vs. land training datasets
percentiles = pre_flood_img.reduceRegion(
    reducer=ee.Reducer.percentile([5, 95]),
    geometry=analysis_zone,
    scale=100,
    maxPixels=1e7
)

water_threshold = ee.Number(percentiles.get('VV_p5'))   # Darkest 5% of pixels
land_threshold = ee.Number(percentiles.get('VV_p95'))  # Brightest 5% of pixels

# 4. Generate Balanced Training Points
water_training = pre_flood_img.updateMask(pre_flood_img.lte(water_threshold)) \
    .sample(region=analysis_zone, scale=30, numPixels=100, geometries=True) \
    .map(lambda f: f.set('landcover', 0))

land_training = pre_flood_img.updateMask(pre_flood_img.gte(land_threshold)) \
    .sample(region=analysis_zone, scale=30, numPixels=100, geometries=True) \
    .map(lambda f: f.set('landcover', 1))

training_features = water_training.merge(land_training)

# 5. Train Random Forest Classifier
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    minLeafPopulation=2
).train(
    features=training_features,
    classProperty='landcover',
    inputProperties=['VV']
)

# 6. Run Machine Learning Classification and Extract Flood Mask
pre_classified = pre_flood_img.classify(classifier)
post_classified = post_classified_img = post_flood_img.classify(classifier)

# Flood Mask logic: Was Land (1) before, turned into Water (0) after
flood_mask = pre_classified.eq(1).And(post_classified.eq(0))

# 7. Calculate Total Flooded Area in Square Kilometers
flood_area_image = flood_mask.multiply(ee.Image.pixelArea())
stats = flood_area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=analysis_zone,
    scale=30,
    maxPixels=1e9
)

# --- THE FIX IS RIGHT HERE ---
# We fetch 'sum' instead of 'VV' because ee.Reducer.sum() renames the key output!
flooded_m2 = stats.get('sum')
flooded_km2 = ee.Number(flooded_m2).divide(1e6)

print("Spatial Metrics Calculation Complete:")
print(f"Total Flooded Area within the 50km radius: {flooded_km2.getInfo():.2f} sq km")

In [35]:
import folium

print("Bypassing server limits. Initializing dynamic web map layer charts...")

# 1. Define a central interactive map widget tracking your Rajanpur target spot
# Center on the specific coordinates where our model found the 422.88 sq km flood footprint
map_center = [29.1044, 70.3250]
interactive_map = folium.Map(location=map_center, zoom_start=10, tiles='OpenStreetMap')

# 2. Structure the cloud-based image rendering color visualizations
# We display the pre-flood radar backscatter layer as a gray-scale base map
pre_flood_vis = pre_flood_img.visualize(min=-25, max=0, palette=['black', 'white'])

# We isolate the random forest flood mask (where value == 1) and color it vibrant royal blue
flood_overlay = flood_mask.updateMask(flood_mask.eq(1))\
                           .visualize(palette=['#0066FF'], min=0, max=1)

# 3. Pull down the interactive tile layers from the remote Google server
try:
    pre_map_id = ee.Image(pre_flood_vis).getMapId()
    folium.TileLayer(
        tiles=pre_map_id['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name='1. Pre-Flood Radar Baseline (Gray)',
        overlay=True,
        control=True
    ).add_to(interactive_map)

    flood_map_id = ee.Image(flood_overlay).getMapId()
    folium.TileLayer(
        tiles=flood_map_id['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name='2. ML Predicted Flood Extent (Blue)',
        overlay=True,
        control=True
    ).add_to(interactive_map)

    # Add a dynamic layer control selector toggle to the top right of the widget layout
    folium.LayerControl().add_to(interactive_map)

    print("📊 SUCCESS! Interactive presentation widget fully built.")
    print("💡 Action: Review your live machine learning results directly inside the map window below!")

except Exception as e:
    print(f"Dynamic render paused: {e}")
    print("Pro Tip: If layers do not populate, click 'Runtime' -> 'Restart session and run all' from the top Colab menu bar to clear cached session variables.")

# Display the map widget right inside your Colab interface layout screen
interactive_map

Bypassing server limits. Initializing dynamic web map layer charts...
Dynamic render paused: Image.visualize: Cannot provide a palette when visualizing more than one band.
Pro Tip: If layers do not populate, click 'Runtime' -> 'Restart session and run all' from the top Colab menu bar to clear cached session variables.


In [36]:
import folium

print("Bypassing server limits. Initializing dynamic web map layer charts...")

# 1. Define a central interactive map widget tracking your Rajanpur target spot
# Center on the specific coordinates where our model found the 422.88 sq km flood footprint
map_center = [29.1044, 70.3250]
interactive_map = folium.Map(location=map_center, zoom_start=10, tiles='OpenStreetMap')

# 2. Structure the cloud-based image rendering color visualizations
# We display the pre-flood radar backscatter layer as a gray-scale base map
pre_flood_vis = pre_flood_img.visualize(min=-25, max=0, palette=['black', 'white'])

# We isolate the random forest flood mask (where value == 1) and color it vibrant royal blue
flood_overlay = flood_mask.updateMask(flood_mask.eq(1))\
                           .visualize(palette=['#0066FF'], min=0, max=1)

# 3. Pull down the interactive tile layers from the remote Google server
try:
    pre_map_id = ee.Image(pre_flood_vis).getMapId()
    folium.TileLayer(
        tiles=pre_map_id['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name='1. Pre-Flood Radar Baseline (Gray)',
        overlay=True,
        control=True
    ).add_to(interactive_map)

    flood_map_id = ee.Image(flood_overlay).getMapId()
    folium.TileLayer(
        tiles=flood_map_id['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name='2. ML Predicted Flood Extent (Blue)',
        overlay=True,
        control=True
    ).add_to(interactive_map)

    # Add a dynamic layer control selector toggle to the top right of the widget layout
    folium.LayerControl().add_to(interactive_map)

    print("📊 SUCCESS! Interactive presentation widget fully built.")
    print("💡 Action: Review your live machine learning results directly inside the map window below!")

except Exception as e:
    print(f"Dynamic render paused: {e}")
    print("Pro Tip: If layers do not populate, click 'Runtime' -> 'Restart session and run all' from the top Colab menu bar to clear cached session variables.")

# Display the map widget right inside your Colab interface layout screen
interactive_map

Bypassing server limits. Initializing dynamic web map layer charts...
Dynamic render paused: Image.visualize: Cannot provide a palette when visualizing more than one band.
Pro Tip: If layers do not populate, click 'Runtime' -> 'Restart session and run all' from the top Colab menu bar to clear cached session variables.


In [37]:
import folium

print("Snapping interactive map zoom directly to the high-intensity flood core...")

# --- THE GEOGRAPHIC ZOOM TWEAK ---
# We center exactly on the flooded riverbanks and bump zoom_start from 10 to 12
flood_core_coords = [29.1044, 70.3250]
interactive_map = folium.Map(location=flood_core_coords, zoom_start=12, tiles='OpenStreetMap')

# Compile our map layers from the server
pre_flood_vis = pre_flood_img.visualize(min=-25, max=0, palette=['black', 'white'])
flood_overlay = flood_mask.updateMask(flood_mask.eq(1)).visualize(palette=['#0066FF'], min=0, max=1)

try:
    pre_map_id = ee.Image(pre_flood_vis).getMapId()
    folium.TileLayer(
        tiles=pre_map_id['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name='1. Pre-Flood Radar Baseline (Gray)',
        overlay=True,
        control=True
    ).add_to(interactive_map)

    flood_map_id = ee.Image(flood_overlay).getMapId()
    folium.TileLayer(
        tiles=flood_map_id['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name='2. ML Predicted Flood Extent (Blue)',
        overlay=True,
        control=True
    ).add_to(interactive_map)

    folium.LayerControl().add_to(interactive_map)
    print("📊 SUCCESS! Map focused securely on the critical impact zone.")

except Exception as e:
    print(f"Dynamic render paused: {e}")

# Display the newly zoomed map layout
interactive_map

Snapping interactive map zoom directly to the high-intensity flood core...
Dynamic render paused: Image.visualize: Cannot provide a palette when visualizing more than one band.
